# JetCobot — Pick & Place desde Jupyter

Ejecutá las celdas **en orden, de arriba hacia abajo**, con `Shift + Enter`.

| Paso | Qué hace | ¿Mueve el brazo? |
|---|---|---|
| 0 | Prepara la carpeta | No |
| 1 | Elige simulación o robot real y conecta | No |
| 2 | Estado del robot | No |
| 3 | Va a OBSERVACIÓN y muestra lo que ve la cámara | **Sí** |
| 4 | Mide la altura de la mesa (`z_min_mm`) | Lo movés vos a mano |
| 5 | Calibración cámara → robot (4 cubos) | Lo movés vos a mano |
| 6 | Recoge **un** color, paso a paso | **Sí** |
| 7 | Ciclo completo (los 4 colores) | **Sí** |
| 8 | Cierra cámara y puerto serie | No |

## ⚠️ Seguridad — leé esto antes
- **Jupyter corre una celda a la vez.** Mientras una celda mueve el brazo, *ninguna otra celda puede ejecutarse*, ni siquiera la de STOP.
- Para frenar: botón **■ (Interrumpir kernel)** de la barra, y después ejecutá la celda **🛑 STOP**.
- **El freno real es el interruptor de energía del robot.** Tené la mano cerca la primera vez.
- Empezá siempre con velocidad baja (15) y modo paso a paso.
- Si otro cuaderno (por ejemplo una demo de Yahboom) está usando la cámara o el robot, **cerrá su kernel** primero: *Kernel → Shut Down*. Si no, la cámara aparece «bloqueada».

## 0 · Carpeta del proyecto
Subí `jetcobot.zip` con el botón **Upload** de Jupyter a tu carpeta personal. Esta celda lo descomprime (si hace falta) y se ubica dentro.

In [ ]:
import os, zipfile
CARPETA = os.path.expanduser('~/jetcobot')   # cambiala si lo pusiste en otro lado
ZIP = os.path.expanduser('~/jetcobot.zip')

if not os.path.isdir(CARPETA) and os.path.exists(ZIP):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(os.path.dirname(CARPETA))
    print('Descomprimido en', CARPETA)

if os.path.exists('pick_place.py'):
    pass                                   # el cuaderno ya está abierto dentro de la carpeta
elif os.path.isdir(CARPETA):
    os.chdir(CARPETA)
else:
    raise SystemExit('No encuentro la carpeta: subí jetcobot.zip a tu carpeta personal')
print('Trabajando en:', os.getcwd())
print(sorted(f for f in os.listdir('.') if not f.startswith('.')))

## 1 · Conectar
Dejá `SIMULAR = True` la primera vez: prueba todo **sin mover nada**. Cuando funcione, cambialo a `False` y volvé a ejecutar esta celda.

In [ ]:
SIMULAR = True      # True = sin hardware · False = robot real
VELOCIDAD = 15      # 1-100; empezá bajo

import json, logging, importlib
import numpy as np, cv2
import pick_place, robot as robot_mod, vision, calibration
for m in (robot_mod, vision, calibration, pick_place):
    importlib.reload(m)   # por si editaste algún .py

# Jupyter acumula handlers si se re-ejecuta la celda: se limpian antes
logging.getLogger().handlers.clear()
logger, csv_file = pick_place.setup_logging()

with open('config.json') as f:
    config = json.load(f)
config['motion']['default_speed'] = VELOCIDAD

# Si se re-ejecuta, primero se cierra lo anterior
try:
    camera.release()
except Exception:
    pass

robot = robot_mod.create_robot(SIMULAR, config)
calibrador = calibration.HomographyCalibrator(config)
camera = detector = None
if not SIMULAR:
    camera = vision.CameraCapture(device=0, width=config['camera']['width'],
                                  height=config['camera']['height'], fps=config['camera']['fps'])
    detector = vision.ColorDetector(config, debug=True)

pp = pick_place.PickPlace(config, robot, camera, detector, calibrador, csv_file, step_by_step=False)
print('Modo:', 'SIMULACIÓN' if SIMULAR else 'ROBOT REAL', '| velocidad', VELOCIDAD)
print('Calibración:', calibrador.get_status())

## 🛑 STOP (emergencia)
Primero **■ Interrumpir kernel**, después esta celda.

In [ ]:
robot.stop()
print('STOP enviado')

## 2 · Estado

In [ ]:
print('Ángulos:', robot.get_angles())
print('Coords :', robot.get_coords())
print('Calibración:', calibrador.get_status())

## 3 · Ver lo que ve la cámara
El brazo va a HOME y luego a OBSERVACIÓN. Los cubos detectados aparecen marcados.

In [ ]:
import matplotlib.pyplot as plt

def foto_y_deteccion():
    """Captura en OBSERVACIÓN, detecta y dibuja. Devuelve {color: (u, v, área)}."""
    if SIMULAR:
        det = pp.detect_colors()
        print('(simulación) detecciones inventadas:', det)
        return det
    for _ in range(5):            # descarta frames viejos del buffer de la cámara
        camera.capture()
    frame = camera.capture()
    if frame is None:
        print('No hay imagen: ¿otro kernel tiene la cámara?')
        return {}
    detector.reset_history()
    det = detector.detect_all_colors(frame)
    img = frame.copy()
    for c, (u, v, a) in det.items():
        cv2.circle(img, (u, v), 6, (0, 255, 0), -1)
        cv2.putText(img, f'{c} ({u},{v})', (u + 8, v), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    plt.figure(figsize=(8, 6)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
    print('Detectados:', det)
    return det

ok = pp.power_on() and pp.go_to_pose('home') and pp.go_to_pose('observation')
print('En OBSERVACIÓN' if ok else 'FALLÓ el movimiento: revisá el log en logs/')
detecciones = foto_y_deteccion() if ok else {}

## 4 · Medir la altura de la mesa (`z_min_mm`)
1. Ejecutá **4a**: los servos quedan **sueltos**. **Sostené el brazo**, porque cae.
2. Bajá la pinza a mano hasta que **toque apenas la mesa**.
3. Ejecutá **4b**: lee la altura, vuelve a trabar los servos y la guarda en `config.json`.

In [ ]:
# 4a — soltar servos (¡sostené el brazo!)
robot.release_all_servos(); print('Servos sueltos: llevá la pinza a la mesa')

In [ ]:
# 4b — leer altura y guardarla
c = robot.get_coords()
robot.power_on()                     # vuelve a trabar los servos
if c:
    z_mesa = round(c[2], 1)
    print('Pinza tocando la mesa en z =', z_mesa, 'mm')
    config['motion']['workspace']['z_min_mm'] = z_mesa
    with open('config.json', 'w') as f:
        json.dump(config, f, indent=2)
    print('Guardado en config.json → z_min_mm =', z_mesa)
else:
    print('No se pudo leer la posición del robot')

## 5 · Calibración cámara → robot
La cámara convierte píxeles en milímetros con una **homografía**. Hacen falta **al menos 4 puntos**, y los 4 cubos sirven de marcadores.

1. Repartí los 4 cubos **separados**, cerca de las esquinas de lo que ve la cámara.
2. **5a**: va a OBSERVACIÓN y toma la foto. Los píxeles de los 4 cubos quedan guardados. ¡No muevas los cubos después!
3. Para cada color: **5b** suelta los servos → llevá a mano la pinza justo **encima del centro** de ese cubo → **5c** con ese color.
4. **5d** calcula y guarda `calibracion.json`. Un error de reproyección menor a ~5 px está bien.

In [ ]:
# 5a — foto de referencia (los cubos NO se mueven después de esto)
calibrador.pixel_points, calibrador.robot_points = [], []
pp.go_to_pose('observation')
pix_ref = foto_y_deteccion()
print('Colores con píxel guardado:', list(pix_ref))

In [ ]:
# 5b — soltar servos (¡sostené el brazo!)
robot.release_all_servos(); print('Servos sueltos: poné la pinza encima del cubo')

In [ ]:
# 5c — registrar UN punto: cambiá COLOR y ejecutá (repetí 5b + 5c por cada color)
COLOR = 'rojo'

def registrar(color, xy_simulado=None):
    if color not in pix_ref:
        print(f'{color} no se vio en la foto 5a'); return
    if xy_simulado is not None:
        x, y = xy_simulado
    else:
        c = robot.get_coords(); robot.power_on()
        if not c:
            print('No se pudo leer la posición'); return
        x, y = c[0], c[1]
    u, v, _ = pix_ref[color]
    calibrador.add_point((u, v), (x, y))
    print(f'{color}: píxel ({u},{v}) → robot ({x:.1f}, {y:.1f}) mm · puntos: {len(calibrador.pixel_points)}')

if SIMULAR:
    # En simulación se inventan 4 posiciones coherentes para probar el flujo
    for col, xy in zip(pix_ref, [(150, -60), (150, 60), (230, 60), (230, -60)]):
        registrar(col, xy)
else:
    registrar(COLOR)

In [ ]:
# 5d — calcular y guardar
if calibrador.compute_homography():
    calibrador.save()
    print('Calibración guardada:', calibrador.get_status())
else:
    print('Faltan puntos o están mal repartidos (necesitás 4, no alineados)')

## 6 · Recoger UN color, paso a paso
Cada movimiento pide **Enter** en la cajita que aparece debajo de la celda. Para cancelar: **■ Interrumpir**.

In [ ]:
COLOR = 'rojo'
pp.step_by_step = not SIMULAR        # en simulación no espera Enter

if calibrador.H is None:
    print('Falta la calibración (paso 5)')
else:
    ok = pp.power_on() and pp.go_to_pose('home') and pp.go_to_pose('observation')
    det = pp.detect_colors() if ok else {}
    if COLOR not in det:
        print(f'{COLOR} no detectado: {det}')
    else:
        exito = pp.pick_and_place_single_color(COLOR, det)
        if exito:
            pp.go_to_pose('home')
        print('✅ Listo' if exito else '❌ Falló: el brazo quedó DETENIDO donde estaba. Revisá logs/')

## 7 · Ciclo completo (los 4 colores)
Solo cuando el paso 6 funcione con cada color. Si un color falla, **el ciclo se detiene** sin más movimientos.

In [ ]:
pp.step_by_step = not SIMULAR        # ponelo en False cuando ya confíes
print('✅ Ciclo completo' if pp.run_full_cycle() else '❌ Ciclo detenido: revisá logs/')

## 8 · Cerrar
Libera la cámara y el puerto serie, para que otros cuadernos puedan usarlos.

In [ ]:
try:
    if camera: camera.release()
    ser = getattr(getattr(robot, 'robot', None), '_serial_port', None)
    if ser: ser.close()
finally:
    print('Cerrado. Para volver a usarlo, ejecutá de nuevo el paso 1.')